# OPDI on OpenSky Network

This notebook demonstrates how to use the OPDI pipeline on the OpenSky Kubernetes cluster.
It connects to the OpenSky S3 bucket, reads state vectors, and runs pipeline steps that
store results as parquet on `s3a://eurocontrol/opdi/`.

## Prerequisites

- Running on the OpenSky JupyterLab environment
- OPDI package installed (`pip install -e .`)
- (Optional) Custom Docker image pushed for distributed mode

## 1. Create a Spark session

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from opdi.utils.spark_helpers import get_spark

# --- Choose one ---

# Non-distributed (runs entirely on the driver, no K8s executors)
spark = get_spark("opensky", app_name="OPDI OpenSky Quickstart")

# Distributed (spawns executors on Kubernetes)
# spark = get_spark("opensky", app_name="OPDI OpenSky Quickstart", distributed=True)

# Distributed with a custom Docker image
# spark = get_spark(
#     "opensky",
#     app_name="OPDI OpenSky Quickstart",
#     distributed=True,
#     container_image="<your-registry>/opdi-spark:v4.1.1",
# )

spark

## 2. Read state vectors directly from S3

In [ ]:
# Read a single hour partition
SRC = "s3a://opensky-hdfs-backup/tables_v4/state_vectors/hour=1771952400"
df = spark.read.parquet(SRC)

print("=== Schema ===")
df.printSchema()

print("=== Sample ===")
df.show(5)

## 3. Read a full day of state vectors

In [ ]:
from datetime import datetime

BASE_PATH = "s3a://opensky-hdfs-backup/tables_v4/state_vectors"

# August 1 2025 (24 hours)
day_start = int(datetime(2025, 8, 1).timestamp())
hour_timestamps = list(range(day_start, day_start + 24 * 3600, 3600))
paths = [f"{BASE_PATH}/hour={h}" for h in hour_timestamps]

df_day = spark.read.parquet(*paths)
print(f"Row count: {df_day.count():,}")

## 4. Write results to S3

In [ ]:
# Write a sample to the eurocontrol S3 bucket
sample = df.limit(100).drop("serials")
sample.write.mode("overwrite").parquet("s3a://eurocontrol/opdi/example_output")
print("Written to s3a://eurocontrol/opdi/example_output")

## 5. Run pipeline steps with StorageManager

All pipeline steps use a `StorageManager` that automatically reads/writes
S3 parquet instead of Iceberg tables when in the `opensky` environment.

### 5a. Ingest the OpenSky aircraft database

In [ ]:
from opdi.config import OPDIConfig
from opdi.ingestion.osn_aircraft_db import AircraftDatabaseIngestion

config = OPDIConfig.for_environment("opensky")

acdb = AircraftDatabaseIngestion(spark, config)
acdb.create_table_if_not_exists()   # no-op in S3 mode
count = acdb.ingest(mode="overwrite")
print(f"Ingested {count} aircraft records to s3a://eurocontrol/opdi/osn_aircraft_db/")

### 5b. Ingest OurAirports reference data

In [ ]:
from opdi.ingestion.ourairports import OurAirportsIngestion

oa = OurAirportsIngestion(spark, config)
oa.create_tables()   # no-op in S3 mode
stats = oa.ingest_all()
stats

### 5c. Verify written data by reading it back

In [ ]:
from opdi.utils.storage import StorageManager

storage = StorageManager(spark, config)

# Read back the aircraft database we just wrote
df_acdb = storage.read_table("osn_aircraft_db")
print(f"Aircraft DB rows: {df_acdb.count():,}")
df_acdb.show(5)

# Read back an OurAirports table
df_airports = storage.read_table("oa_airports")
print(f"Airports rows: {df_airports.count():,}")
df_airports.show(5)

## 6. Count state vectors per day (August 2025)

In [ ]:
from datetime import datetime
from pyspark.sql import functions as F

AUG_START = int(datetime(2025, 8, 1).timestamp())
AUG_END = int(datetime(2025, 9, 1).timestamp())

hours = list(range(AUG_START, AUG_END, 3600))
paths = [f"{BASE_PATH}/hour={h}" for h in hours]
print(f"Reading {len(paths)} hour partitions...")

df_aug = spark.read.parquet(*paths)

daily_counts = (
    df_aug
    .withColumn("date", F.to_date(F.from_unixtime(F.col("time"))))
    .groupBy("date")
    .agg(F.count("*").alias("state_vector_count"))
    .orderBy("date")
)

daily_counts.show(31, truncate=False)

In [ ]:
# Save results to S3
daily_counts.write.mode("overwrite").csv(
    "s3a://eurocontrol/opdi/august_2025_daily_counts", header=True
)
print("Saved to s3a://eurocontrol/opdi/august_2025_daily_counts/")

## 7. Plot daily counts

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pdf = daily_counts.toPandas()
pdf["date"] = pd.to_datetime(pdf["date"])

plt.figure(figsize=(14, 5))
plt.plot(pdf["date"], pdf["state_vector_count"], marker="o", linewidth=2, markersize=5)
plt.xlabel("Date")
plt.ylabel("State vectors")
plt.title("State Vectors per Day - August 2025")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Clean up

In [ ]:
spark.stop()